<a href="https://colab.research.google.com/github/harshPandde/harshPandde/blob/main/Project__assistent_using_combination_of_unstructure_and_structure_data_sources.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.6 MB/s eta 0:00:00


In [ ]:
import os

os.makedirs("data", exist_ok=True)

policy_text = """
Company Leave Policy:
Employees are entitled to 20 paid leaves annually.
Casual leave limit is 10 days.
Sick leave limit is 10 days.

Work From Home Policy:
Employees may work from home twice a week with manager approval.

Salary Policy:
Salaries are credited on the last working day of every month.
"""

with open("data/company_policy.txt", "w") as f:
    f.write(policy_text)

In [ ]:
import sqlite3
import os

os.makedirs("database", exist_ok=True)

conn = sqlite3.connect("database/company.db")
cursor = conn.cursor()

# 🔥 DROP TABLE IF EXISTS (THIS FIXES EVERYTHING)
cursor.execute("DROP TABLE IF EXISTS employees")

cursor.execute("""
CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    salary INTEGER,
    leaves_taken INTEGER
)
""")

employees_data = [
    (1, "Amit", "Marketing", 50000, 5),
    (2, "Sara", "Engineering", 80000, 2),
    (3, "Raj", "Marketing", 55000, 7),
    (4, "Priya", "HR", 45000, 4),
    (5, "John", "Engineering", 90000, 1)
]

cursor.executemany("INSERT INTO employees VALUES (?, ?, ?, ?, ?)", employees_data)

conn.commit()
conn.close()

print("Database reset and recreated successfully.")

Database reset and recreated successfully.


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

class RAGEngine:

    def __init__(self, file_path):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.text_chunks = self.load_data(file_path)
        self.index = self.create_index()

    def load_data(self, file_path):
        with open(file_path, "r") as f:
            text = f.read()
        return [chunk.strip() for chunk in text.split("\n") if chunk.strip()]

    def create_index(self):
        embeddings = self.model.encode(self.text_chunks)
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatL2(dimension)
        index.add(np.array(embeddings))
        return index

    def retrieve(self, query, k=2):
        query_embedding = self.model.encode([query])
        distances, indices = self.index.search(np.array(query_embedding), k)
        return [self.text_chunks[i] for i in indices[0]]

In [ ]:
rag = RAGEngine("data/company_policy.txt")
print("RAG system ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

RAG system ready.


In [ ]:
class SQLEngine:

    def __init__(self, db_path):
        self.db_path = db_path

    def execute(self, query):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute(query)
        result = cursor.fetchall()
        conn.close()
        return result

In [ ]:
sql_engine = SQLEngine("database/company.db")
print("SQL engine ready.")

SQL engine ready.


In [ ]:
def generate_sql(question):
    prompt = f"""
You are an SQL expert.

Table: employees(id, name, department, salary, leaves_taken)

Convert to SQLite SQL query:
{question}
SQL:
"""

    response = llm(prompt, max_new_tokens=60)
    generated = response[0]["generated_text"]

    # Remove prompt from output if repeated
    sql_query = generated.split("SQL:")[-1].strip()

    return sql_query

In [ ]:
def route_question(question):
    sql_keywords = ["average", "count", "salary", "leaves", "total", "department"]

    if any(word in question.lower() for word in sql_keywords):
        return "sql"
    return "rag"

In [ ]:
def process(question):

    route = route_question(question)

    if route == "sql":
        sql_query = generate_sql(question)
        print("Generated SQL:", sql_query)

        try:
            result = sql_engine.execute(sql_query)
            return f"Result: {result}"
        except Exception as e:
            return f"SQL Error: {e}"

    else:
        context = rag.retrieve(question)
        return "Relevant Information:\n" + "\n".join(context)

In [ ]:

while True:
    question = input("\nAsk a question (type 'exit' to stop): ")

    if question.lower() == "exit":
        break

    answer = process(question)
    print(answer)


Ask a question (type 'exit' to stop): exit


In [ ]:
rag.retrieve("What is leave policy?")

['Company Leave Policy:', 'Casual leave limit is 10 days.']

In [ ]:
generate_sql("What is average salary in Marketing?")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'n March 1997 If One Thring Out By Eager For Any Profit But Still Has A Plenty Of House On'

In [ ]:
from transformers import pipeline

llm = pipeline(
    task="text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

LLM loaded successfully.


In [ ]:
print(type(llm))

<class 'transformers.pipelines.text_generation.TextGenerationPipeline'>
